# Example: Complete UMEP workflow using pymdurs and solweig

This example demonstrates how to:
1. Collect urban data using pymdurs (DEM, buildings, vegetation)
2. Process data for UMEP analysis (DSM, CDSM)
3. Calculate Sky View Factor (SVF) using solweig
4. Optionally run SOLWEIG for thermal comfort analysis

Inspired by: https://github.com/UMEP-dev/solweig/blob/dev/demos/athens-demo.py

Required dependencies (install separately):
```bash
pip install geopandas rasterio pyproj pillow
pip install "solweig @ git+https://github.com/UMEP-dev/solweig.git@main"
```

Note: On Apple Silicon (ARM64), solweig may require the x86_64 target:
```bash
rustup target add x86_64-apple-darwin
```


In [ ]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio
import solweig
from rasterio.features import rasterize, shapes
from rasterio.transform import from_bounds
from shapely.geometry import box, shape

import pymdurs

# Resolve project paths (notebook may run from notebooks/ or project root)
_cwd = Path.cwd()
if (_cwd / "notebooks" / "umep_workflow.ipynb").exists():
    PROJECT_ROOT = _cwd
    NOTEBOOKS_DIR = _cwd / "notebooks"
elif (_cwd / "umep_workflow.ipynb").exists():
    NOTEBOOKS_DIR = _cwd
    PROJECT_ROOT = _cwd.parent
elif (_cwd.parent / "examples").exists():
    PROJECT_ROOT = _cwd.parent
    NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
else:
    PROJECT_ROOT = _cwd
    NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

EXAMPLES_DIR = PROJECT_ROOT / "examples"
OUTPUT_DIR = PROJECT_ROOT / "output" / "umep_workflow"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

from utils import (  # noqa: E402
    create_solweig_preview_gifs,
    create_solweig_preview_pngs,
    warp_clip_raster,
)

# Optional umepr / umep for SVF and wall heights
try:
    from umepr import svf

    HAS_UMEPR = True
except ImportError:
    HAS_UMEPR = False
    print("umepr package not available. SVF calculation will be skipped.")
    print("   Install with: pip install 'solweig @ git+https://github.com/UMEP-dev/solweig.git@main'")

try:
    from umep import wall_heightaspect_algorithm  # noqa: F401

    HAS_UMEP = True
except ImportError:
    HAS_UMEP = False
    print("umep package not available. Wall height generation will be skipped.")

print("HAS_UMEPR:", HAS_UMEPR)
print("HAS_UMEP:", HAS_UMEP)
print("GPU available:", solweig.is_gpu_available())
print("Compute backend:", solweig.get_compute_backend())

# COSIA color to class mapping
TABLE_COLOR_COSIA = {
    "Bâtiment": "#ce7079",
    "Zone imperméable": "#a6aab7",
    "Zone perméable": "#987752",
    "Piscine": "#62d0ff",
    "Serre": "#b9e2d4",
    "Sol nu": "#bbb096",
    "Surface eau": "#3375a1",
    "Neige": "#e9effe",
    "Conifère": "#216e2e",
    "Feuillu": "#4c9129",
    "Coupe": "#e48e4d",
    "Broussaille": "#b5c335",
    "Pelouse": "#8cd76a",
    "Culture": "#decf55",
    "Terre labourée": "#d0a349",
    "Vigne": "#b08290",
    "Autre": "#222222",
}

COSIA_TO_UMEP = {
    "Bâtiment": 2,
    "Zone imperméable": 1,
    "Zone perméable": 6,
    "Piscine": 7,
    "Serre": 1,
    "Sol nu": 6,
    "Surface eau": 7,
    "Neige": 7,
    "Conifère": 6,
    "Feuillu": 6,
    "Coupe": 5,
    "Broussaille": 5,
    "Pelouse": 5,
    "Culture": 5,
    "Terre labourée": 6,
    "Vigne": 5,
    "Autre": 1,
}

UMEP_LABELS = {
    1: "Paved",
    2: "Building",
    3: "Evergreen Trees",
    4: "Deciduous Trees",
    5: "Grass",
    6: "Bare Soil",
    7: "Water",
}


def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i : i + 2], 16) for i in (0, 2, 4))


def geodataframe_to_tif_with_metadata(
    gdf: gpd.GeoDataFrame,
    output_tif: str,
    column: str = "type",
    resolution: float = 1.0,
):
    print(f"\nConverting GeoDataFrame to TIF (column={column}, resolution={resolution} m)")
    if len(gdf) == 0:
        raise ValueError("GeoDataFrame is empty, cannot create a raster")

    bounds = gdf.total_bounds
    width = int((bounds[2] - bounds[0]) / resolution)
    height = int((bounds[3] - bounds[1]) / resolution)
    if width <= 0 or height <= 0:
        raise ValueError(f"Invalid dimensions: width={width}, height={height}")

    transform = from_bounds(bounds[0], bounds[1], bounds[2], bounds[3], width, height)
    shapes_iter = ((geom, value) for geom, value in zip(gdf.geometry, gdf[column]))
    raster = rasterize(
        shapes=shapes_iter,
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype=np.uint8,
        all_touched=False,
    )

    with rasterio.open(
        output_tif,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=raster.dtype,
        crs=gdf.crs,
        transform=transform,
        compress="lzw",
        nodata=0,
    ) as dst:
        dst.write(raster, 1)
        dst.update_tags(
            description="COSIA Land Cover Classification (UMEP format)",
            resolution=f"{resolution}m",
            classes=str(UMEP_LABELS),
        )

    print(f"File saved: {output_tif}")
    return raster


def vectorize_cosia_raster(cosia_tiff_path: str):
    print(f"\nVectorizing COSIA raster: {cosia_tiff_path}")
    rgb_to_class = {hex_to_rgb(color): land_class for land_class, color in TABLE_COLOR_COSIA.items()}

    with rasterio.open(cosia_tiff_path) as src:
        image = src.read()
        transform = src.transform
        crs = src.crs
        combined = (
            (image[0].astype(np.uint32) << 16)
            + (image[1].astype(np.uint32) << 8)
            + image[2].astype(np.uint32)
        )
        results = shapes(combined, transform=transform)
        geoms = []
        rgb_values = []
        for geom, value in results:
            value_int = int(value) if isinstance(value, (float, np.floating)) else value
            r = (value_int >> 16) & 255
            g = (value_int >> 8) & 255
            b = value_int & 255
            geoms.append(shape(geom))
            rgb_values.append((r, g, b))

    gdf = gpd.GeoDataFrame({"rgb": rgb_values, "geometry": geoms}, crs=crs)

    def match_color(rgb):
        min_dist = float("inf")
        best = "Autre"
        for target_rgb, land_class in rgb_to_class.items():
            dist = sum((a - b) ** 2 for a, b in zip(rgb, target_rgb))
            if dist < min_dist:
                min_dist = dist
                best = land_class
        return best

    gdf["landcover_class"] = gdf["rgb"].apply(match_color)
    gdf["color"] = gdf["landcover_class"].map(TABLE_COLOR_COSIA)
    gdf["type"] = gdf["landcover_class"].map(COSIA_TO_UMEP)
    gdf = gdf.drop(columns=["rgb"])
    print(f"Vectorization complete: {len(gdf)} polygons")
    return gdf


def fill_nodata_nan(tif_path, nodata_fallback=-9999):
    path = Path(tif_path)
    if not path.exists():
        return
    with rasterio.open(path, "r+") as src:
        nd = src.nodata if src.nodata is not None else nodata_fallback
        for i in range(1, src.count + 1):
            band = src.read(i)
            if np.issubdtype(band.dtype, np.floating) and np.any(np.isnan(band)):
                band = np.where(np.isnan(band), nd, band)
                src.write(band, i)
        src.nodata = nd


In [ ]:
# Configuration — Bordeaux
output_path = OUTPUT_DIR
output_path.mkdir(parents=True, exist_ok=True)
output_folder_str = str(output_path)

# Bounding box (Bordeaux, France) — WGS84 EPSG:4326
bbox_wgs84 = (-0.5833492801, 44.8457876761, -0.5737192696, 44.8509319773)

minx, miny, maxx, maxy = bbox_wgs84
geom_wgs84 = box(minx, miny, maxx, maxy)
gdf_bbox = gpd.GeoDataFrame(geometry=[geom_wgs84], crs="EPSG:4326")
gdf_bbox = gdf_bbox.to_crs(2154)
bbox_2154 = tuple(gdf_bbox.total_bounds)

working_crs = 2154

# Data files (absolute paths)
EPW_PATH = EXAMPLES_DIR / "bordeaux_2025.epw"
PHYSICS_PATH = EXAMPLES_DIR / "physics_defaults.json"
MATERIALS_PATH = EXAMPLES_DIR / "default_materials.json"
CONFIG_PATH = output_path / "my_config.json"

# SOLWEIG location — center of the Bordeaux bbox
location = solweig.Location(latitude=44.8484, longitude=-0.5785, utc_offset=1)

print(f"Bounding box: {bbox_wgs84}")
print(f"Working CRS: EPSG:{working_crs}")
print(f"Output folder: {output_folder_str}")
print(f"EPW: {EPW_PATH}")


## COSIA — Step 1: Download COSIA from IGN API

In [ ]:
print("=" * 60)
print("Step 1: Downloading COSIA from IGN API...")
print("=" * 60)

cosia = pymdurs.geometric.Cosia(output_path=output_folder_str)
cosia.set_bbox(*bbox_wgs84)
cosia.set_crs(working_crs)

print("Downloading COSIA from IGN API...")
cosia = cosia.run_ign()

cosia_tiff_path = cosia.get_path_save_tiff()
print(f"COSIA downloaded: {cosia_tiff_path}")

if os.path.exists(cosia_tiff_path):
    size = os.path.getsize(cosia_tiff_path) / (1024 * 1024)
    print(f"File size: {size:.2f} MB")


## COSIA — Step 2: Vectorize COSIA raster

In [ ]:
print("=" * 60)
print("Step 2: Vectorizing COSIA raster...")
print("=" * 60)

gdf = vectorize_cosia_raster(cosia_tiff_path)

landcover_shp = output_path / "cosia_landcover.shp"
gdf.to_file(landcover_shp, driver="ESRI Shapefile")
print(f"Shapefile saved: {landcover_shp}")


## COSIA — Step 3: Convert to UMEP format and rasterize

In [ ]:
print("=" * 60)
print("Step 3: Converting to UMEP format and rasterizing...")
print("=" * 60)

gdf = gdf.to_crs(working_crs)
gdf_valid = gdf[gdf.geometry.notna()].copy()
print(f"{len(gdf_valid)} valid geometries out of {len(gdf)} total")

landcover_tif = output_path / "landcover.tif"
landcover_source = landcover_tif
raster = geodataframe_to_tif_with_metadata(
    gdf=gdf_valid,
    output_tif=str(landcover_tif),
    column="type",
    resolution=1.0,
)
print(f"UMEP landcover raster: {landcover_tif}")


## UMEP — Step 1: Collect DEM from IGN API

In [ ]:
print("=" * 60)
print("Step 1: Collecting DEM from IGN API...")
print("=" * 60)

dem = pymdurs.geometric.Dem(output_path=output_folder_str)
dem.set_bbox(*bbox_wgs84)
dem.set_crs(working_crs)
dem = dem.run()

dem_source = output_path / "DEM.tif"
print(f"DEM collected and saved to: {dem_source}")


## UMEP — Step 2: Load LiDAR data from IGN WFS service

In [ ]:
dsm_source = output_path / "DSM.tif"
cdsm_source = output_path / "CDSM.tif"

if not dsm_source.exists():
    print("=" * 60)
    print("Step 2: Loading LiDAR data from IGN WFS service...")
    print("=" * 60)

    lidar = pymdurs.geometric.Lidar(output_path=output_folder_str)
    lidar.set_bbox(*bbox_wgs84)
    lidar.set_crs(working_crs)

    print("Bounding box set")
    print(f"CRS: {lidar.geo_core.epsg}")

    print("Generating CDSM from vegetation and water classes...")
    lidar.run(file_name="CDSM.tif", classification_list=[3, 4, 5, 9])
    print("CDSM generated")

    print("Generating DSM from ground and buildings classes...")
    dsm_output_path = lidar.run(file_name="DSM.tif", classification_list=[2, 6])
    print(f"DSM GeoTIFF saved to: {dsm_output_path}")

    if os.path.exists(dsm_output_path):
        size = os.path.getsize(dsm_output_path) / (1024 * 1024)
        print(f"DSM GeoTIFF file size: {size:.2f} MB")
else:
    print("DSM already exists, skipping LiDAR download")


## UMEP — Step 3: Warp and clip rasters using mask

In [ ]:
print("=" * 60)
print("Step 3: Warping and clipping rasters with mask...")
print("=" * 60)

mask_shp_path = output_path / "mask.shp"
dem_clip_path = output_path / "DEM_clip.tif"
dsm_clip_path = output_path / "DSM_clip.tif"
cdsm_clip_path = output_path / "CDSM_clip.tif"
landcover_clip_path = output_path / "landcover_clip.tif"

if mask_shp_path.exists():
    clip_targets = [
        ("DEM", dem_source, dem_clip_path),
        ("DSM", dsm_source, dsm_clip_path),
        ("CDSM", cdsm_source, cdsm_clip_path),
        ("Landcover", landcover_source, landcover_clip_path),
    ]
    for label, src, dst in clip_targets:
        if src.exists():
            warp_clip_raster(src, dst, mask_shp_path)
            print(f"{label} clipped to: {dst}")
        elif label == "Landcover":
            print("Warning: landcover.tif missing")
else:
    print("Warning: Mask shapefile not found, skipping clipping")

dsm_path = dsm_clip_path
cdsm_path = cdsm_clip_path
dem_tiff_path = dem_clip_path
lc_path = landcover_clip_path


## Step 3a: Filling DSM NoData with DEM values

In [ ]:
if dem_clip_path.exists() and dsm_clip_path.exists():
    print("Step 3a: Filling DSM NoData with DEM values...")

    with rasterio.open(dem_clip_path) as dem_src:
        dem_data = dem_src.read(1)
        dem_nodata = dem_src.nodata or -99999.0

    with rasterio.open(dsm_clip_path) as dsm_src:
        dsm_data = dsm_src.read(1)
        dsm_profile = dsm_src.profile.copy()
        dsm_nodata = dsm_src.nodata or 0

    dsm_invalid = (dsm_data == dsm_nodata) | np.isnan(dsm_data) | (dsm_data == 0)
    dem_valid = (dem_data != dem_nodata) & ~np.isnan(dem_data)
    fill_mask = dsm_invalid & dem_valid
    filled_count = int(np.sum(fill_mask))

    if filled_count > 0:
        dsm_data[fill_mask] = dem_data[fill_mask]
        dsm_profile.update(nodata=-9999.0)
        with rasterio.open(dsm_clip_path, "w", **dsm_profile) as dst:
            dst.write(dsm_data, 1)
        print(f"Filled {filled_count} DSM NoData pixels ({filled_count / dsm_data.size * 100:.2f}%)")
    else:
        print("No DSM NoData pixels to fill")
else:
    print("Skipping Step 3a — clipped DEM/DSM not available")


## Step 4: Calculate Sky View Factor (SVF) using umepr

In [ ]:
print("=" * 60)
print("Step 4: Calculating Sky View Factor (SVF) using umepr...")
print("=" * 60)

if not HAS_UMEPR:
    print("Skipping SVF calculation - umepr not available")
elif dsm_path.exists():
    with rasterio.open(dsm_path) as dsm_src:
        bounds = dsm_src.bounds
        total_extents = [bounds.left, bounds.bottom, bounds.right, bounds.top]

    print(f"Using raster bounds for SVF: {total_extents}")
    svf_output_dir = output_path / "svf"
    svf_output_dir.mkdir(parents=True, exist_ok=True)

    try:
        svf.generate_svf(
            dsm_path=str(dsm_path),
            bbox=total_extents,
            out_dir=str(svf_output_dir),
            cdsm_path=str(cdsm_path) if cdsm_path.exists() else None,
            trunk_ratio_perc=25,
            trans_veg_perc=3,
            use_tiled_loading=False,
        )
        print(f"SVF calculation complete! Output in: {svf_output_dir}")
    except Exception as e:
        print(f"SVF calculation failed: {type(e).__name__}: {e}")
else:
    print("Skipping SVF calculation - DSM not available")


## Step 5: Generate wall heights for SOLWEIG

In [ ]:
if HAS_UMEP and dsm_path.exists():
    print("=" * 60)
    print("Step 5: Generating wall heights for SOLWEIG...")
    print("=" * 60)

    with rasterio.open(dsm_path) as dsm_src:
        bounds = dsm_src.bounds
        total_extents = [bounds.left, bounds.bottom, bounds.right, bounds.top]

    try:
        walls_output_dir = output_path / "walls"
        walls_output_dir.mkdir(parents=True, exist_ok=True)
        wall_heightaspect_algorithm.generate_wall_hts(
            dsm_path=str(dsm_path),
            bbox=total_extents,
            out_dir=str(walls_output_dir),
        )
        print(f"Wall heights generated! Output in: {walls_output_dir}")
    except Exception as e:
        print(f"Wall height generation failed: {type(e).__name__}: {e}")
else:
    print("Skipping Step 5 - umep not available or DSM missing")


## Step 6: Run SOLWEIG for thermal comfort analysis

In [ ]:
# Step 1: Prepare surface data
if not (dsm_path.exists() and lc_path.exists()):
    raise RuntimeError("DSM and landcover required for SOLWEIG — run previous steps first")

for rast in (dsm_path, cdsm_path, dem_tiff_path):
    if rast.exists():
        fill_nodata_nan(rast)

print("=" * 60)
print("Step 6: Running SOLWEIG for thermal comfort analysis...")
print("=" * 60)

surface = solweig.SurfaceData.prepare(
    dsm=str(dsm_path),
    working_dir=str(output_path / "working"),
    cdsm=str(cdsm_path),
    pixel_size=1.0,
    land_cover=str(lc_path),
    cdsm_relative=False,
)
print("Surface data prepared")


In [ ]:
weather_list = solweig.Weather.from_epw(
    str(EPW_PATH),
    start="2025-07-01 07:00",
    end="2025-07-01 19:00",
)
physics = solweig.load_physics(str(PHYSICS_PATH))
materials = solweig.load_materials(str(MATERIALS_PATH))
config = solweig.ModelConfig.defaults()
config.save(str(CONFIG_PATH))

results = solweig.calculate(
    surface=surface,
    weather=weather_list,
    location=location,
    physics=physics,
    materials=materials,
    human=solweig.HumanParams(
        abs_k=0.65,
        abs_l=0.97,
        weight=70,
        height=1.65,
        posture="standing",
    ),
    use_anisotropic_sky=True,
    conifer=False,
    output_dir=str(output_path),
    outputs=["tmrt", "shadow", "utci"],
)
print("SOLWEIG run complete!")
print(results.report())


In [ ]:
# Plot timeseries (Ta, Tmrt, UTCI, radiation, sun exposure over time)
results.plot()


In [ ]:
# Visualize summary grids
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

im0 = axes[0, 0].imshow(results.tmrt_mean, cmap="viridis")
axes[0, 0].set_title("Mean Tmrt (°C)")
plt.colorbar(im0, ax=axes[0, 0])

im1 = axes[0, 1].imshow(results.utci_mean, cmap="inferno")
axes[0, 1].set_title("Mean UTCI (°C)")
plt.colorbar(im1, ax=axes[0, 1])

im2 = axes[0, 2].imshow(results.sun_hours, cmap="YlOrRd")
axes[0, 2].set_title("Sun hours")
plt.colorbar(im2, ax=axes[0, 2])

im3 = axes[1, 0].imshow(results.tmrt_day_mean, cmap="inferno")
axes[1, 0].set_title("Mean daytime Tmrt (°C)")
plt.colorbar(im3, ax=axes[1, 0])

im4 = axes[1, 1].imshow(results.tmrt_night_mean, cmap="cool")
axes[1, 1].set_title("Mean nighttime Tmrt (°C)")
plt.colorbar(im4, ax=axes[1, 1])

threshold = sorted(results.utci_hours_above.keys())[0]
im5 = axes[1, 2].imshow(results.utci_hours_above[threshold], cmap="Reds")
axes[1, 2].set_title(f"UTCI hours > {threshold}°C")
plt.colorbar(im5, ax=axes[1, 2])

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(
    f"SOLWEIG Summary — {len(results)} timesteps ({results.n_daytime} day, {results.n_nighttime} night)"
)
plt.tight_layout()
plt.show()


In [ ]:
# Load and inspect run metadata
metadata = solweig.load_run_metadata(output_path / "run_metadata.json")
print("Run metadata loaded:")
print(f"  Timestamp: {metadata['run_timestamp']}")
print(f"  SOLWEIG version: {metadata['solweig_version']}")
print(
    f"  Location: {metadata['location']['latitude']:.2f}°N, {metadata['location']['longitude']:.2f}°E"
)
print(f"  Human posture: {metadata.get('human', {}).get('posture', 'default (standing)')}")
print(f"  Anisotropic sky: {metadata['parameters']['use_anisotropic_sky']}")
print(f"  Weather timesteps: {metadata['timeseries']['timesteps']}")
print(f"  Date range: {metadata['timeseries']['start']} to {metadata['timeseries']['end']}")

create_solweig_preview_gifs(output_path)
create_solweig_preview_pngs(output_path)
print("Preview GIFs and PNGs created")


## Summary

In [ ]:
print("=" * 60)
print("UMEP workflow complete!")
print("=" * 60)
print(f"All outputs saved to: {output_folder_str}")
print("\nGenerated files:")
if dsm_path.exists():
    print(f"  - DSM: {dsm_path}")
if cdsm_path.exists():
    print(f"  - CDSM: {cdsm_path}")
if dem_tiff_path.exists():
    print(f"  - DEM: {dem_tiff_path}")
if lc_path.exists():
    print(f"  - Landcover: {lc_path}")
if (output_path / "svf").exists():
    print(f"  - SVF: {output_path / 'svf'}")
if (output_path / "walls").exists():
    print(f"  - Wall heights: {output_path / 'walls'}")
